In [0]:
# =============================================================================
# Smart ERP – Product Silver Layer Simulation  (Databricks / Delta Lake)
# Catalog  : smart_odoo
# Targets  :
#     silver.product_category   (10 rows)
#     silver.product_template   (100 rows : 70 laptops + 30 accessories)
#     silver.product_product    (100 rows : 1-to-1 with template)
#
# FIXES vs previous version
# ──────────────────────────
# FIX 1 – 100 unique product names (was 33).  Each product_id maps to
#          exactly one name via a hard-coded catalogue — no random.choice
#          collision risk.  Names follow real-world brand/model patterns.
# FIX 2 – Realistic EGP prices.  Laptop prices match Egyptian market:
#          i3 ~15K, i5 ~22K, i7 ~35K, i9 ~50K, Apple chips 35K–90K.
#          Accessories reduced from 800-6000 to 300-5000 by category.
#          Cost is still 85-95% of list price (unchanged logic).
#
# IDEMPOTENCY GUARANTEES
# ──────────────────────
# ✅ Fixed seed          → identical dates on every run
# ✅ Hard-coded catalogue → names/prices never change between runs
# ✅ MERGE on PK         → re-running never creates duplicate rows
# ✅ Table created once  → saveAsTable("errorifexists") on first run only
# ✅ dwh_loaded_at       → refreshed to current_timestamp() on every upsert
# ✅ Temp-views dropped  → no stale-view collision across runs
# ✅ Pre-flight asserts  → catches logic errors before touching Delta
#
# Product rules
# ─────────────
#   product_id  1 –  18  → Dell    laptops  (categ_id 4)
#   product_id 19 –  35  → HP      laptops  (categ_id 5)
#   product_id 36 –  52  → Lenovo  laptops  (categ_id 6)
#   product_id 53 –  70  → Apple   laptops  (categ_id 7)
#   product_id 71 –  77  → Mouse       accessories (categ_id 8)
#   product_id 78 –  83  → Keyboard    accessories (categ_id 9)
#   product_id 84 – 100  → Other       accessories (categ_id 10)
#   Date range           : 2025-01-01 → 2026-04-22
# =============================================================================

# ── 0. Imports ────────────────────────────────────────────────────────────────
import random
from datetime import datetime, timedelta, timezone

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, BooleanType,
    DoubleType, TimestampType,
)

# Fixed seed → identical dates on every run
SEED = 42
random.seed(SEED)

spark = SparkSession.builder.getOrCreate()

# ── 1. Constants ──────────────────────────────────────────────────────────────
CATALOG   = "smart_odoo"
SCHEMA    = "silver"
SOURCE_DB = "python_ingest"

START_DATE = datetime(2025, 1, 1)
END_DATE   = datetime(2026, 6, 15)

# ── 2. Product Catalogue — 100 unique products ────────────────────────────────
#
# Each entry is fully explicit: (product_id, categ_id, product_type, name, list_price)
# No random.choice on names → guaranteed unique across all runs.
# Prices in EGP, realistic Egyptian market rates (2025).

LAPTOP_CATALOGUE = [
    # ── Dell XPS  (1–9) ── high-end ultrabook line ──────────────────────────
    ( 1, 4, "Laptop", "Dell XPS 13 9310 Core i5 11th Gen",          22500.0),
    ( 2, 4, "Laptop", "Dell XPS 13 9310 Core i7 11th Gen",          32000.0),
    ( 3, 4, "Laptop", "Dell XPS 13 9310 Core i9 11th Gen",          47000.0),
    ( 4, 4, "Laptop", "Dell XPS 13 9315 Core i5 12th Gen",          24000.0),
    ( 5, 4, "Laptop", "Dell XPS 13 9315 Core i7 12th Gen",          34500.0),
    ( 6, 4, "Laptop", "Dell XPS 15 9510 Core i5 11th Gen",          28000.0),
    ( 7, 4, "Laptop", "Dell XPS 15 9510 Core i7 11th Gen",          38000.0),
    ( 8, 4, "Laptop", "Dell XPS 15 9510 Core i9 11th Gen",          52000.0),
    ( 9, 4, "Laptop", "Dell XPS 15 9520 Core i7 12th Gen",          40000.0),
    # ── Dell Inspiron  (10–14) ── mainstream / value ─────────────────────────
    (10, 4, "Laptop", "Dell Inspiron 14 3420 Core i3 12th Gen",     14500.0),
    (11, 4, "Laptop", "Dell Inspiron 14 3420 Core i5 12th Gen",     19500.0),
    (12, 4, "Laptop", "Dell Inspiron 15 3520 Core i5 12th Gen",     20500.0),
    (13, 4, "Laptop", "Dell Inspiron 15 3520 Core i7 12th Gen",     28500.0),
    (14, 4, "Laptop", "Dell Inspiron 16 Plus 7620 Core i7 12th Gen",31000.0),
    # ── Dell Latitude  (15–18) ── business line ──────────────────────────────
    (15, 4, "Laptop", "Dell Latitude 5430 Core i5 12th Gen",        26000.0),
    (16, 4, "Laptop", "Dell Latitude 5530 Core i7 12th Gen",        36000.0),
    (17, 4, "Laptop", "Dell Latitude 7430 Core i7 12th Gen",        42000.0),
    (18, 4, "Laptop", "Dell Latitude 7530 Core i7 12th Gen",        44000.0),
    # ── HP ZBook  (19–23) ── mobile workstation ───────────────────────────────
    (19, 5, "Laptop", "HP ZBook Fury 15 G8 Core i7 11th Gen",       38000.0),
    (20, 5, "Laptop", "HP ZBook Fury 15 G8 Core i9 11th Gen",       53000.0),
    (21, 5, "Laptop", "HP ZBook Fury 16 G9 Core i7 12th Gen",       42000.0),
    (22, 5, "Laptop", "HP ZBook Fury 16 G9 Core i9 12th Gen",       57000.0),
    (23, 5, "Laptop", "HP ZBook Studio 16 G9 Core i7 12th Gen",     46000.0),
    # ── HP EliteBook  (24–29) ── business line ───────────────────────────────
    (24, 5, "Laptop", "HP EliteBook 840 G8 Core i5 11th Gen",       27000.0),
    (25, 5, "Laptop", "HP EliteBook 840 G8 Core i7 11th Gen",       36000.0),
    (26, 5, "Laptop", "HP EliteBook 840 G9 Core i5 12th Gen",       29000.0),
    (27, 5, "Laptop", "HP EliteBook 840 G9 Core i7 12th Gen",       38000.0),
    (28, 5, "Laptop", "HP EliteBook 860 G9 Core i7 12th Gen",       40000.0),
    (29, 5, "Laptop", "HP EliteBook 1040 G9 Core i7 12th Gen",      45000.0),
    # ── HP Pavilion  (30–35) ── mainstream / value ────────────────────────────
    (30, 5, "Laptop", "HP Pavilion 14 Core i3 12th Gen",            14000.0),
    (31, 5, "Laptop", "HP Pavilion 14 Core i5 12th Gen",            19000.0),
    (32, 5, "Laptop", "HP Pavilion 15 Core i5 12th Gen",            20000.0),
    (33, 5, "Laptop", "HP Pavilion 15 Core i7 12th Gen",            27500.0),
    (34, 5, "Laptop", "HP Pavilion 16 Core i5 12th Gen",            21500.0),
    (35, 5, "Laptop", "HP Pavilion 16 Core i7 12th Gen",            29000.0),
    # ── Lenovo ThinkPad  (36–44) ── business / pro ───────────────────────────
    (36, 6, "Laptop", "Lenovo ThinkPad T14 Gen 2 Core i5 11th Gen", 25000.0),
    (37, 6, "Laptop", "Lenovo ThinkPad T14 Gen 2 Core i7 11th Gen", 34000.0),
    (38, 6, "Laptop", "Lenovo ThinkPad T14 Gen 3 Core i5 12th Gen", 27000.0),
    (39, 6, "Laptop", "Lenovo ThinkPad T14 Gen 3 Core i7 12th Gen", 36000.0),
    (40, 6, "Laptop", "Lenovo ThinkPad T16 Gen 1 Core i5 12th Gen", 29000.0),
    (41, 6, "Laptop", "Lenovo ThinkPad T16 Gen 1 Core i7 12th Gen", 38500.0),
    (42, 6, "Laptop", "Lenovo ThinkPad X1 Carbon Gen 10 Core i5 12th Gen", 39000.0),
    (43, 6, "Laptop", "Lenovo ThinkPad X1 Carbon Gen 10 Core i7 12th Gen", 49000.0),
    (44, 6, "Laptop", "Lenovo ThinkPad X1 Extreme Gen 5 Core i7 12th Gen", 55000.0),
    # ── Lenovo IdeaPad  (45–52) ── mainstream / value ─────────────────────────
    (45, 6, "Laptop", "Lenovo IdeaPad 3 14 Core i3 12th Gen",       13500.0),
    (46, 6, "Laptop", "Lenovo IdeaPad 3 14 Core i5 12th Gen",       18500.0),
    (47, 6, "Laptop", "Lenovo IdeaPad 3 15 Core i5 12th Gen",       19500.0),
    (48, 6, "Laptop", "Lenovo IdeaPad 5 14 Core i5 12th Gen",       22000.0),
    (49, 6, "Laptop", "Lenovo IdeaPad 5 15 Core i5 12th Gen",       23000.0),
    (50, 6, "Laptop", "Lenovo IdeaPad 5 15 Core i7 12th Gen",       31000.0),
    (51, 6, "Laptop", "Lenovo IdeaPad Slim 5 16 Core i5 12th Gen",  24500.0),
    (52, 6, "Laptop", "Lenovo IdeaPad Slim 5 16 Core i7 12th Gen",  33000.0),
    # ── Apple MacBook Air  (53–57) ────────────────────────────────────────────
    (53, 7, "Laptop", "Apple MacBook Air 13 M1",                    35000.0),
    (54, 7, "Laptop", "Apple MacBook Air 13 M2",                    45000.0),
    (55, 7, "Laptop", "Apple MacBook Air 15 M2",                    52000.0),
    (56, 7, "Laptop", "Apple MacBook Air 13 M3",                    50000.0),
    (57, 7, "Laptop", "Apple MacBook Air 15 M3",                    57000.0),
    # ── Apple MacBook Pro M2  (58–63) ─────────────────────────────────────────
    (58, 7, "Laptop", "Apple MacBook Pro 14 M2",                    55000.0),
    (59, 7, "Laptop", "Apple MacBook Pro 14 M2 Pro",                65000.0),
    (60, 7, "Laptop", "Apple MacBook Pro 14 M2 Max",                88000.0),
    (61, 7, "Laptop", "Apple MacBook Pro 16 M2",                    62000.0),
    (62, 7, "Laptop", "Apple MacBook Pro 16 M2 Pro",                75000.0),
    (63, 7, "Laptop", "Apple MacBook Pro 16 M2 Max",                95000.0),
    # ── Apple MacBook Pro M3  (64–70) ─────────────────────────────────────────
    (64, 7, "Laptop", "Apple MacBook Pro 14 M3",                    58000.0),
    (65, 7, "Laptop", "Apple MacBook Pro 14 M3 Pro",                72000.0),
    (66, 7, "Laptop", "Apple MacBook Pro 14 M3 Max",                95000.0),
    (67, 7, "Laptop", "Apple MacBook Pro 16 M3",                    68000.0),
    (68, 7, "Laptop", "Apple MacBook Pro 16 M3 Pro",                85000.0),
    (69, 7, "Laptop", "Apple MacBook Pro 16 M3 Max",               110000.0),
    (70, 7, "Laptop", "Apple MacBook Pro 13 M2",                    48000.0),
]

ACCESSORY_CATALOGUE = [
    # ── Mouse  (71–77)  categ 8 ───────────────────────────────────────────────
    (71, 8,  "Accessory", "Logitech MX Master 3S Wireless Mouse",        850.0),
    (72, 8,  "Accessory", "Logitech M705 Marathon Wireless Mouse",        450.0),
    (73, 8,  "Accessory", "Razer DeathAdder V2 Wired Gaming Mouse",       700.0),
    (74, 8,  "Accessory", "Razer Basilisk V3 Wireless Mouse",             950.0),
    (75, 8,  "Accessory", "Microsoft Arc Bluetooth Mouse",                550.0),
    (76, 8,  "Accessory", "HP 930 Creator Wireless Mouse",                480.0),
    (77, 8,  "Accessory", "Dell MS5320W Multi-Device Wireless Mouse",     420.0),
    # ── Keyboard  (78–83)  categ 9 ────────────────────────────────────────────
    (78, 9,  "Accessory", "Logitech MX Keys Advanced Wireless Keyboard",  1200.0),
    (79, 9,  "Accessory", "Logitech K780 Multi-Device Wireless Keyboard",  800.0),
    (80, 9,  "Accessory", "Razer BlackWidow V3 Mechanical Keyboard",      1100.0),
    (81, 9,  "Accessory", "Microsoft Sculpt Ergonomic Wireless Keyboard",  900.0),
    (82, 9,  "Accessory", "HP 950 Dual Mode Wireless Keyboard",            650.0),
    (83, 9,  "Accessory", "Dell KB700 Multi-Device Wireless Keyboard",     600.0),
    # ── USB Hub  (84–88)  categ 10 ────────────────────────────────────────────
    (84, 10, "Accessory", "Anker 341 USB-C Hub 7-in-1",                   750.0),
    (85, 10, "Accessory", "Anker 565 USB-C Hub 11-in-1",                 1400.0),
    (86, 10, "Accessory", "Belkin Connect USB-C Hub 6-in-1",              950.0),
    (87, 10, "Accessory", "Ugreen USB-C Hub Revodok 9-in-1",              850.0),
    (88, 10, "Accessory", "CalDigit TS4 Thunderbolt 4 Dock",             4500.0),
    # ── External SSD  (89–92)  categ 10 ──────────────────────────────────────
    (89, 10, "Accessory", "Samsung T7 Portable SSD 500GB",               1100.0),
    (90, 10, "Accessory", "Samsung T7 Portable SSD 1TB",                 1900.0),
    (91, 10, "Accessory", "WD My Passport SSD 1TB",                      1700.0),
    (92, 10, "Accessory", "SanDisk Extreme Portable SSD 1TB",            1800.0),
    # ── Headset  (93–96)  categ 10 ────────────────────────────────────────────
    (93, 10, "Accessory", "Jabra Evolve2 55 Wireless Headset",           4200.0),
    (94, 10, "Accessory", "Logitech Zone Wired USB Headset",             1500.0),
    (95, 10, "Accessory", "HP Poly Voyager Focus 2 UC Headset",          3200.0),
    (96, 10, "Accessory", "Dell Pro Stereo Headset UC350",                900.0),
    # ── Webcam  (97–100)  categ 10 ────────────────────────────────────────────
    (97, 10, "Accessory", "Logitech C920 HD Pro Webcam 1080p",            950.0),
    (98, 10, "Accessory", "Logitech C925e Business Webcam 1080p",        1100.0),
    (99, 10, "Accessory", "Microsoft Modern Webcam 1080p",                750.0),
   (100, 10, "Accessory", "Razer Kiyo Pro Webcam 1080p",                 1300.0),
]

ALL_PRODUCTS = LAPTOP_CATALOGUE + ACCESSORY_CATALOGUE   # 100 entries

# ── 3. Schemas ────────────────────────────────────────────────────────────────
categ_schema = StructType([
    StructField("categ_id",                          IntegerType(),  False),
    StructField("parent_id",                         IntegerType(),  True),
    StructField("parent_name",                       StringType(),   True),
    StructField("name",                              StringType(),   True),
    StructField("complete_name",                     StringType(),   True),
    StructField("parent_path",                       StringType(),   True),
    StructField("product_properties_definition",     StringType(),   True),
    StructField("property_account_income_categ_id",  IntegerType(),  True),
    StructField("property_account_income_categ_name",StringType(),   True),
    StructField("property_account_expense_categ_id", IntegerType(),  True),
    StructField("property_account_expense_categ_name",StringType(),  True),
    StructField("created_at",                        TimestampType(), True),
    StructField("updated_at",                        TimestampType(), True),
    StructField("dwh_loaded_at",                     TimestampType(), True),
    StructField("dwh_source_db",                     StringType(),   True),
])

tmpl_schema = StructType([
    StructField("product_tmpl_id",              IntegerType(),  False),
    StructField("categ_id",                     IntegerType(),  True),
    StructField("categ_name",                   StringType(),   True),
    StructField("uom_id",                       IntegerType(),  True),
    StructField("uom_name",                     StringType(),   True),
    StructField("company_id",                   IntegerType(),  True),
    StructField("company_name",                 StringType(),   True),
    StructField("color",                        IntegerType(),  True),
    StructField("type",                         StringType(),   True),
    StructField("service_tracking",             StringType(),   True),
    StructField("default_code",                 StringType(),   True),
    StructField("name",                         StringType(),   True),
    StructField("description",                  StringType(),   True),
    StructField("description_purchase",         StringType(),   True),
    StructField("description_sale",             StringType(),   True),
    StructField("product_properties",           StringType(),   True),
    StructField("list_price",                   DoubleType(),   True),
    StructField("volume",                       DoubleType(),   True),
    StructField("weight",                       DoubleType(),   True),
    StructField("sale_ok",                      BooleanType(),  True),
    StructField("purchase_ok",                  BooleanType(),  True),
    StructField("active",                       BooleanType(),  True),
    StructField("can_image_1024_be_zoomed",     BooleanType(),  True),
    StructField("has_configurable_attributes",  BooleanType(),  True),
    StructField("is_favorite",                  BooleanType(),  True),
    StructField("property_account_income_id",   IntegerType(),  True),
    StructField("property_account_income_name", StringType(),   True),
    StructField("property_account_expense_id",  IntegerType(),  True),
    StructField("property_account_expense_name",StringType(),   True),
    StructField("service_type",                 StringType(),   True),
    StructField("expense_policy",               StringType(),   True),
    StructField("invoice_policy",               StringType(),   True),
    StructField("sale_line_warn_msg",           StringType(),   True),
    StructField("created_at",                   TimestampType(), True),
    StructField("updated_at",                   TimestampType(), True),
    StructField("dwh_loaded_at",                TimestampType(), True),
    StructField("dwh_source_db",                StringType(),   True),
])

product_schema = StructType([
    StructField("product_id",                       IntegerType(),  False),
    StructField("product_tmpl_id",                  IntegerType(),  True),
    StructField("product_tmpl_name",                StringType(),   True),
    StructField("default_code",                     StringType(),   True),
    StructField("barcode",                          StringType(),   True),
    StructField("combination_indices",              StringType(),   True),
    StructField("standard_price",                   DoubleType(),   True),
    StructField("volume",                           DoubleType(),   True),
    StructField("weight",                           DoubleType(),   True),
    StructField("active",                           BooleanType(),  True),
    StructField("can_image_variant_1024_be_zoomed", BooleanType(),  True),
    StructField("is_favorite",                      BooleanType(),  True),
    StructField("is_in_selected_section_of_order",  BooleanType(),  True),
    StructField("created_at",                       TimestampType(), True),
    StructField("updated_at",                       TimestampType(), True),
    StructField("dwh_loaded_at",                    TimestampType(), True),
    StructField("dwh_source_db",                    StringType(),   True),
])

# ── 4. Build product_category rows ───────────────────────────────────────────
CATEGORIES_RAW = [
    (1,  None, None,           "All Products", "All Products",                          "1/"),
    (2,  1,    "All Products", "Laptops",      "All Products / Laptops",                "1/2/"),
    (3,  1,    "All Products", "Accessories",  "All Products / Accessories",            "1/3/"),
    (4,  2,    "Laptops",      "Dell",         "All Products / Laptops / Dell",         "1/2/4/"),
    (5,  2,    "Laptops",      "HP",           "All Products / Laptops / HP",           "1/2/5/"),
    (6,  2,    "Laptops",      "Lenovo",       "All Products / Laptops / Lenovo",       "1/2/6/"),
    (7,  2,    "Laptops",      "Apple",        "All Products / Laptops / Apple",        "1/2/7/"),
    (8,  3,    "Accessories",  "Mouse",        "All Products / Accessories / Mouse",    "1/3/8/"),
    (9,  3,    "Accessories",  "Keyboard",     "All Products / Accessories / Keyboard", "1/3/9/"),
    (10, 3,    "Accessories",  "Other",        "All Products / Accessories / Other",    "1/3/10/"),
]

now = datetime.now(timezone.utc).replace(tzinfo=None)

categ_rows = []
for cid, pid, pname, name, cname, path in CATEGORIES_RAW:
    categ_rows.append((
        cid, pid, pname, name, cname, path,
        None,           # product_properties_definition
        None, None,     # property_account_income_categ_id / name
        None, None,     # property_account_expense_categ_id / name
        now, now, now, SOURCE_DB,
    ))

categ_name_by_id = {r[0]: r[4] for r in categ_rows}   # categ_id → complete_name

# ── 5. SKU builder ────────────────────────────────────────────────────────────
def _sku(pid: int, name: str) -> str:
    """
    Deterministic SKU from product_id + first two words of name.
    Format: PRD-{id:03d}-{BRAND_ABBR}
    Examples: PRD-001-DEL, PRD-053-APL, PRD-071-LOG
    """
    abbr = name.split()[0][:3].upper()
    return f"PRD-{pid:03d}-{abbr}"

# ── 6. Build product_template + product_product rows ─────────────────────────
def _rdate() -> datetime:
    """Random date in [START_DATE, END_DATE] using the global fixed seed."""
    return START_DATE + timedelta(
        days=random.randint(0, (END_DATE - START_DATE).days)
    )

def _cost(price: float) -> float:
    """Cost = 85–95% of list price, rounded to 2 decimals."""
    return round(price * random.uniform(0.85, 0.95), 2)

def _weight(product_type: str) -> float:
    """Realistic weight range by product type."""
    if product_type == "Laptop":
        return round(random.uniform(1.2, 2.8), 2)
    return round(random.uniform(0.1, 1.2), 2)

tmpl_rows    = []
product_rows = []

for entry in ALL_PRODUCTS:
    pid, categ_id, product_type, name, list_price = entry
    cost    = _cost(list_price)
    weight  = _weight(product_type)
    sku     = _sku(pid, name)
    created = _rdate()

    tmpl_rows.append((
        pid,                            # product_tmpl_id
        categ_id,                       # categ_id
        categ_name_by_id[categ_id],     # categ_name (full path)
        1,  "Unit(s)",                  # uom_id / uom_name
        1,  "Alpha Corp",               # company_id / company_name
        0,                              # color
        product_type,                   # type
        None,                           # service_tracking
        sku,                            # default_code
        name,                           # name
        None, None, None, None,         # description fields
        list_price,                     # list_price
        0.0,                            # volume
        weight,                         # weight
        True, True, True,               # sale_ok, purchase_ok, active
        False, False, False,            # can_image, has_configurable, is_favorite
        None, None, None, None,         # account income/expense ids/names
        None, None, None, None,         # service_type, expense_policy, invoice_policy, warn
        created, created, now, SOURCE_DB,
    ))
    product_rows.append((
        pid,                            # product_id
        pid,                            # product_tmpl_id (1-to-1)
        name,                           # product_tmpl_name
        sku,                            # default_code
        None,                           # barcode
        None,                           # combination_indices
        cost,                           # standard_price
        0.0,                            # volume
        weight,                         # weight
        True,                           # active
        False,                          # can_image_variant_1024_be_zoomed
        False,                          # is_favorite
        False,                          # is_in_selected_section_of_order
        created, created, now, SOURCE_DB,
    ))

# ── 7. Create Spark DataFrames ────────────────────────────────────────────────
df_categ    = spark.createDataFrame(categ_rows,    schema=categ_schema)
df_template = spark.createDataFrame(tmpl_rows,     schema=tmpl_schema)
df_product  = spark.createDataFrame(product_rows,  schema=product_schema)

# ── 8. Pre-flight assertions ──────────────────────────────────────────────────
def _preflight():
    cids  = [r[0]  for r in categ_rows]
    tids  = [r[0]  for r in tmpl_rows]
    pids  = [r[0]  for r in product_rows]
    names = [r[11] for r in tmpl_rows]   # index 11 = name column

    assert len(cids)  == len(set(cids)),  "❌ Duplicate categ_id"
    assert len(tids)  == len(set(tids)),  "❌ Duplicate product_tmpl_id"
    assert len(pids)  == len(set(pids)),  "❌ Duplicate product_id"
    assert len(names) == len(set(names)), "❌ Duplicate product name"

    assert len(categ_rows)   == 10,  f"❌ Expected 10 categories, got {len(categ_rows)}"
    assert len(tmpl_rows)    == 100, f"❌ Expected 100 templates,  got {len(tmpl_rows)}"
    assert len(product_rows) == 100, f"❌ Expected 100 products,   got {len(product_rows)}"

    laptops = [r for r in tmpl_rows if r[0] <= 70]
    acc     = [r for r in tmpl_rows if r[0] >= 71]
    assert all(r[8] == "Laptop"    for r in laptops), "❌ Type mismatch in laptops"
    assert all(r[8] == "Accessory" for r in acc),     "❌ Type mismatch in accessories"

    for r in tmpl_rows:
        price = r[16]   # list_price
        cost  = product_rows[r[0] - 1][6]   # standard_price (product_id is 1-indexed)
        assert price > 0,          f"❌ Non-positive list_price on {r[11]}"
        assert cost  > 0,          f"❌ Non-positive standard_price on {r[11]}"
        assert cost  < price,      f"❌ Cost >= list_price on {r[11]}"

        created = r[-4]   # created_at position
        assert START_DATE <= created <= END_DATE, f"❌ Date out of range: {created}"

    print("✅  Pre-flight assertions passed")

_preflight()

# ── 9. Idempotent MERGE writer ────────────────────────────────────────────────
def upsert_silver(df, table: str, pk: str) -> None:
    """
    Run 1  → creates the Delta table via saveAsTable (errorifexists).
    Run 2+ → MERGE on <pk>:
               MATCHED     → update all columns; dwh_loaded_at = current_timestamp()
               NOT MATCHED → insert new row
    Temp view is dropped after each call to keep the session clean.
    """
    full_table = f"{CATALOG}.{SCHEMA}.{table}"
    view_name  = f"_sim_src_{table}"

    table_exists = spark.catalog.tableExists(full_table)

    if not table_exists:
        (df.write
           .format("delta")
           .mode("errorifexists")
           .option("overwriteSchema", "false")
           .saveAsTable(full_table))
        print(f"  [created]  {full_table}  →  {df.count()} rows")
        return

    df.createOrReplaceTempView(view_name)

    other_cols = [c for c in df.columns if c not in (pk, "dwh_loaded_at")]
    set_parts  = [f"t.{c} = s.{c}" for c in other_cols]
    set_parts.append("t.dwh_loaded_at = current_timestamp()")
    set_clause = ",\n            ".join(set_parts)

    spark.sql(f"""
        MERGE INTO {full_table} AS t
        USING {view_name}       AS s
        ON t.{pk} = s.{pk}

        WHEN MATCHED THEN UPDATE SET
            {set_clause}

        WHEN NOT MATCHED THEN INSERT *
    """)

    spark.catalog.dropTempView(view_name)
    print(f"  [merged]   {full_table}  →  upserted {df.count()} rows")


# ── 10. Execute ───────────────────────────────────────────────────────────────
spark.sql(f"USE CATALOG {CATALOG}")

print("\nWriting silver layer …")
upsert_silver(df_categ,    "product_category", "categ_id")
upsert_silver(df_template, "product_template", "product_tmpl_id")
upsert_silver(df_product,  "product_product",  "product_id")

print(f"""
✅  Product simulation complete – smart_odoo.silver populated.
    product_category : {df_categ.count()}   rows
    product_template : {df_template.count()} rows  (Dell=18 | HP=17 | Lenovo=17 | Apple=18 | Accessories=30)
    product_product  : {df_product.count()} rows
""")
